In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

# Load Dataset

In [ ]:
df_train = pd.read_csv('data/titanic_train.csv')

In [ ]:
df_train.info()

In [ ]:
df_train = pd.read_csv('data/titanic_train.csv')
target = df_train['Survived']
df_train = df_train.drop(columns='Survived')

In [ ]:
df_test = pd.read_csv('data/titanic_test.csv')

In [ ]:
df_train.head()

In [ ]:
df_train.info()

Variable...................Definition\
Survived...................survival,  0 = No, 1 = Yes\
Pclass.....................ticket class, 1 = 1st, 2 = 2nd, 3 = 3rd\
Sex........................sex \
Age........................age in years (fraction)\
Sibsp......................number of siblings (irmãos, irmãs)/spouses aboard the Titanic	\
Parch......................number of parents / children aboard the Titanic	\
Ticket.....................número do ticket de passagem	\
Fare.......................valor da passagem	\
Cabin......................número da cabine	\
Embarked...................porto de embarque;	C = Cherbourg, Q = Queenstown, S = Southampton\

# Feature Engineering

## Unique Values

In [ ]:
# Quantidade de unique values em cada coluna.
count = {}

for i in df_train.columns:
    count[i] = len(df_train[i].unique())

count

In [ ]:
len(df_train.columns), len(df_test.columns)

In [ ]:
count_same = 0

for cx, cy in zip(df_train.columns, df_test.columns):
    if cx != cy:
        count_same+=1
    
count_same

## Features

In [ ]:
df_train['PassengerId']

In [ ]:
df_train['Pclass'].value_counts()

In [ ]:
df_train['Name']

Extrair os títulos

In [ ]:
df_train['Title'] = df_train['Name'].str.extract(pat = r'([A-Za-z]+\.)')
df_train['Title']

In [ ]:
df_train['Title'].value_counts()

In [ ]:
df_test['Title'] = df_test['Name'].str.extract(pat = r'([A-Za-z]+\.)')

In [ ]:
df_train['SibSp'].value_counts()

In [ ]:
df_train['Parch'].value_counts()

In [ ]:
df_train['Ticket'].value_counts()

In [ ]:
df_train['Fare'].value_counts()

In [ ]:
df_train['Cabin']

Somente passageiros da primeira classe tem número da Cabine\
A primeira letra representa o Deck\
Extrair a primeira letra

In [ ]:
# Excluir Cabin
df_train['Deck'] = df_train['Cabin'].str.extract(pat = r'([A-Za-z]+)')
df_train['Deck']

In [ ]:
df_test['Deck'] = df_test['Cabin'].str.extract(pat = r'([A-Za-z]+)')

In [ ]:
df_train['Family_Size'] = df_train['SibSp'] + df_train['Parch']
df_test['Family_Size'] = df_test['SibSp'] + df_test['Parch']

## Drop Useless Variables

In [ ]:
count

In [ ]:
df_train = df_train.drop(labels=['Name', 'PassengerId', 'Ticket', 'Cabin'], axis=1)
df_train.head()

In [ ]:
df_test = df_test.drop(labels=['Name', 'PassengerId', 'Ticket', 'Cabin'], axis=1)
df_test.head()

## Deals with Missing Values

In [ ]:
df_train.isnull().sum(axis = 0)

In [ ]:
df_test.isnull().sum(axis = 0)

In [ ]:
df_train['Deck'] = df_train['Deck'].fillna('Unknown')
df_test['Deck'] = df_train['Deck'].fillna('Unknown')

In [ ]:
mean_age_treino = df_train['Age'].mean(skipna=True)
mean_age_teste = df_test['Age'].mean(skipna=True)

In [ ]:
df_train['Age'] = df_train['Age'].fillna(value = mean_age_treino)
df_test['Age'] = df_test['Age'].fillna(value = mean_age_teste)

In [ ]:
df_train.isna().sum(axis=0)

In [ ]:
df_test.isna().sum(axis=0)

In [ ]:
median_fare = df_train['Fare'].median(skipna=True)

In [ ]:
df_test['Fare'] = df_test['Fare'].fillna(value = median_fare)

In [ ]:
df_train['Embarked']

In [ ]:
df_test.isna().sum(axis=0)

In [ ]:
df_train.isna().sum(axis=0)

Drop remaining missing values

In [ ]:
idx = np.nonzero(df_train['Embarked'].isna())[0]
idx

In [ ]:
df_train_final = df_train.drop(labels=idx).copy()
target_final = target.drop(labels=idx)

In [ ]:
df_test_final = df_test.copy()

In [ ]:
df_train_final.sample(5)

In [ ]:
df_test_final.sample(5)

## Encode Categorical Variables

In [ ]:
one_hot_vars = ['Pclass', 'Sex', 'SibSp', 'Parch', 'Embarked', 'Title', 'Deck', 'Family_Size']
one_hot_vars

In [ ]:
def encode_categorical(train, test, categoric_vars):

    m = []
    for i in categoric_vars:
    
        cat = set(train[i].unique())
        cat.update(test[i].unique())

        d = {}
        v = 0
        
        for k in cat:
            d[k] = v
            v +=1
            
        train[i] = train[i].map(d)
        test[i] = test[i].map(d)
        m.append(list(d.values()))

    return train, test, m

In [ ]:
data_treino, data_teste, m = encode_categorical(df_train_final, df_test_final, one_hot_vars)

In [ ]:
one_hot_enc = OneHotEncoder(categories= m, sparse_output = False)

In [ ]:
one_hot_enc.fit(data_treino[one_hot_vars])

In [ ]:
d_treino = one_hot_enc.transform(data_treino[one_hot_vars])

In [ ]:
d_treino.shape

In [ ]:
df_treino_final = np.c_[d_treino, data_treino['Fare'], data_treino['Age']]

In [ ]:
d_teste = one_hot_enc.transform(data_teste[one_hot_vars])

In [ ]:
d_teste.shape

In [ ]:
df_teste_final = np.c_[d_teste, data_teste['Fare'], data_teste['Age']]

In [ ]:
df_treino_final.shape, df_teste_final.shape

In [ ]:
X = df_treino_final

In [ ]:
y = target_final

In [ ]:
y.shape

# Split

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size = 0.3)

In [ ]:
df_numpy_train = np.c_[X_train, y_train]
df_numpy_valid = np.c_[X_valid, y_valid]

# Random Forest API

In [ ]:
class Node:  
    
    def __init__(self,
                 df = None,
                 feature_index = None,
                 threshold = None,
                 gain = None, 
                 left_node = None,
                 right_node = None,
                 number_of_elements = None,
                 gini = None,
                 composition = None,
                 label = None,
                 name = None):
        
        self.df = df
        self.feature_index = feature_index
        self.threshold = threshold
        self.info_gain = gain
        self.left = left_node
        self.right = right_node
        self.number_of_elements = number_of_elements
        self.gini = gini
        self.composition = composition
        self.name = name
        
        'leaf node'
        self.label = label
         
    def __repr__(self):
        return self.name

In [ ]:
def get_node_composition(y):
    a, b = np.unique(y, return_counts=True)
    c = pd.DataFrame(data = np.c_[a,b], columns=['Classes', 'Count'])
    return c

In [ ]:
def assign_label(y):
    y = list(y)
    return max(set(y), key=y.count)

In [ ]:
def calc_gini(y):

    counts = np.bincount(y.astype(np.int32))
    probs = counts[counts > 0] / len(y)

    return 1 - np.sum(probs**2)

In [ ]:
def calc_info_gain(y, left, right):
    h_t = calc_gini(y)
    
    p_left = len(left) / len(y)
    p_right = len(right) / len(y)
    h_s = calc_gini(left[:,-1]) * p_left + calc_gini(right[:,-1]) * p_right
    
    return h_t - h_s

In [ ]:
def get_split(df, feature, threshold):
    s_left = np.array(df[df[:,feature] <= threshold])
    s_right = np.array(df[df[:,feature] > threshold])
    return s_left, s_right

In [ ]:
def get_best_criteria(df, name):
    
    X, y = df[:,:-1], df[:,-1]
    
    n_samples, n_features = np.shape(X)
    gini = calc_gini(y)
    comp = get_node_composition(y)
    name = name
    
    node = Node(df = df, number_of_elements = n_samples, gini = gini, composition = comp, name = name)
    
    max_info_gain = -float('inf')
    
    n_sub_features = int(np.sqrt(n_features))

    features = np.random.choice(n_features, n_sub_features, replace=False)

    for feature_index in features:
            
            for threshold in np.unique(X[:,feature_index]):
                
                left_node, right_node = get_split(df, feature_index, threshold)
                
                if len(left_node) > 5 and len(right_node) > 5:
                    
                    curr_info_gain = calc_info_gain(y, left_node, right_node)
                    
                    if curr_info_gain > max_info_gain:
                        
                        max_info_gain = curr_info_gain
                        node.feature_index = feature_index
                        node.threshold = threshold
                        node.info_gain = max_info_gain
        
    return node

In [ ]:
def build_tree(df, name, max_depth=15):
     
    #n_samples = np.shape(df[:,:-1])[0]

    curr_depth=0
    y = df[:,-1]
    
    node = get_best_criteria(df, name)
    
    ### Leafs ###

    # Pure Node
    if len(np.unique(y)) == 1:
        node.label = y[0]
        return node
      
    # Max Depth
    if curr_depth >= max_depth:
        node.label = assign_label(y)
        return node
           
    if node.info_gain is None:
        node.label = assign_label(y)
        return node
    
    left, right = get_split(df, node.feature_index, node.threshold)
    node.left = build_tree(left, node.name + '.left' , curr_depth+1)
    node.right = build_tree(right, node.name + '.right', curr_depth+1)
        
    return node # Leaf

In [ ]:
def generate_forest(df, n_trees, name='root'):
    lista = []

    n_samples = len(df)

    for _ in range(n_trees):
        indices = np.random.choice(a=n_samples, size=n_samples, replace=True)
        subset_df = df[indices, :]
        lista.append(build_tree(subset_df, name))

    return lista

In [ ]:
def predict(forest, sample) -> int:
    lista = []
    
    def predict_sample(tree, sample):
            if tree.label is not None:
                lista.append(int(tree.label))

            else:
                
                if sample[tree.feature_index] <= tree.threshold:
                    return predict_sample(tree.left, sample)
                
                else:
                    sample[tree.feature_index] > tree.threshold
                    return predict_sample(tree.right, sample)
    
    for tree in forest:
        predict_sample(tree, sample)
    
    class_predicted = max(set(lista), key=lista.count)    
    return class_predicted

In [ ]:
def predict_dataset(forest, X):
    y_pred = [predict(forest, x) for x in X]
    
    return np.array(y_pred)

# Generate Forest with Training Data

In [ ]:
forest = generate_forest(df_numpy_train, 100)

# Predict

## Predict Train Dataset

In [ ]:
y_pred_train = predict_dataset(forest, df_numpy_train[:,:-1])

In [ ]:
accuracy_score(y_train, y_pred_train)

## Predict Valid Dataset

In [ ]:
y_pred_valid = predict_dataset(forest, df_numpy_valid[:,:-1])

In [ ]:
accuracy_score(y_valid, y_pred_valid)

## Compare to Scikit-Learn

In [ ]:
clf = RandomForestClassifier(n_estimators=100,
                             min_samples_leaf=5,
                             max_depth=15)

In [ ]:
clf.fit(X_train, y_train)
y_pred_valid_sk = clf.predict(X_valid)

In [ ]:
accuracy_score(y_valid, y_pred_valid_sk)

# End